### Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is — and therefore, how accurate the LLM’s final answer will be.

That’s where Query Expansion / Enhancement comes in.

#### 🎯 What is Query Enhancement?
Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.
It is especially useful when:

- The original query is short, ambiguous, or under-specified
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants

In [1]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

C:\Users\harsh\AppData\Local\Temp\ipykernel_13180\669806456.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
d:\Github_Codes\Ultimate RAG Bootcamp\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [37]:
## step1 : Load and split the dataset
loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)

In [38]:
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [39]:
### step 2: Vector Store
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(chunks,embedding_model)

## step 3:MMR Retriever
retriever=vectorstore.as_retriever(search_type="mmr",search_kwargs={"k":5})
retriever

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2931.81it/s]


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002651DA0C410>, search_type='mmr', search_kwargs={'k': 5})

In [40]:
## step 4 : LLM and Prompt

import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm=init_chat_model("groq:llama-3.3-70b-versatile")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002651DA0D6D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002651DA0E0D0>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [41]:
# Query expansion
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.
Try to keep the new query in close proximity to the original query in terms of the correctness of the topic, if possible.

Original query: "{query}"

Expanded query:
""")

query_expansion_chain=query_expansion_prompt| llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\nTry to keep the new query in close proximity to the original query in terms of the correctness of the topic, if possible.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.

In [42]:
query_expansion_chain.invoke({"query":"Langchain memory"})

'Expanded query: "Langchain memory" OR "Langchain knowledge retention" OR "conversational AI memory" OR "large language model recall" OR "LLM memory architecture" OR "Langchain contextual understanding" OR "chatbot memory management" OR "Langchain information storage" OR "AI language model memory optimization"\n\nAdditional context: This expanded query aims to capture relevant information related to the memory capabilities of Langchain, a framework for building conversational AI models. The added terms and phrases are intended to cover various aspects of memory in Langchain, including knowledge retention, recall, and memory architecture, as well as related concepts in conversational AI and large language models.\n\nTechnical terms:\n\n* LLM: Large Language Model\n* Conversational AI: Artificial intelligence designed to simulate human-like conversations\n* Contextual understanding: The ability of a model to understand the context of a conversation\n* Memory architecture: The design and 

In [43]:
# RAG answering prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.
Context:
{context}

Question: {input}
""")

document_chain=create_stuff_documents_chain(llm=llm,prompt=answer_prompt)

In [44]:
# Step 5: Full RAG pipeline with query expansion
rag_pipeline = (
    RunnableMap({
        "input": lambda x: x["input"],
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
    })
    | document_chain
)

In [45]:
# Step 6: Run query
query = {"input": "What types of memory does LangChain support?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

To improve document retrieval, I'll expand the query by incorporating relevant synonyms, technical terms, and useful context while maintaining the original topic's correctness. Here's the expanded query:

"{'input': 'What types of memory does LangChain support, including short-term, long-term, episodic, semantic, working, and external memory, as well as any specific memory architectures, models, or mechanisms it utilizes, such as key-value stores, relational databases, or neural network-based memory structures?'}"

This expanded query includes:

1. **Relevant synonyms**: Added terms like "short-term," "long-term," "episodic," "semantic," "working," and "external" to cover various aspects of memory.
2. **Technical terms**: Incorporated technical phrases like "key-value stores," "relational databases," and "neural network-based memory structures" to capture specific memory architectures and models.
3. **Useful context**: Provided additional context by mentioning "memory architectures," "

In [46]:
# Step 6: Run query
query = {"input": "CrewAI agents?"}
print(query_expansion_chain.invoke({"query":query}))
response = rag_pipeline.invoke(query)
print("✅ Answer:\n", response)

Expanded query: "{'input': 'CrewAI agents, autonomous crew management systems, artificial intelligence in crew operations, AI-powered crew planning, intelligent crew scheduling, machine learning for crew optimization, automated crew assignment, crew resource management using AI, smart crew allocation'}"

This expanded query adds relevant synonyms and technical terms related to CrewAI agents, such as autonomous crew management systems, artificial intelligence, and machine learning. It also provides useful context by including terms like crew planning, scheduling, optimization, and allocation, which are all relevant to the topic of CrewAI agents. This should improve document retrieval by capturing a wider range of relevant information. 

Alternatively, the query could also be expanded to include related concepts, such as:

* "{'input': 'CrewAI agents, digital crew management, AI-driven crew solutions, crew performance optimization, data analytics for crew operations, predictive modeling 